In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def show(img, title=""):
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    plt.show()

In [ ]:
image = cv2.imread("image.png", cv2.IMREAD_COLOR_RGB)
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

print(gray.shape)
print(len(np.unique(gray)))

show(gray)

In [ ]:
_, B = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
print(np.unique(B))
print(len(np.unique(B)))

show(B)

In [ ]:
def disk(d):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (d, d))

In [ ]:
show(disk(64))

In [ ]:
def make_ring(ksize, width):
    assert width % 2 == 0
    ek = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    inner_el = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize - width, ksize - width))
    inner_el = cv2.copyMakeBorder(inner_el, *(width // 2,)*4, borderType=cv2.BORDER_CONSTANT, value=0)
    return ek - inner_el

In [ ]:
show(make_ring(64, 10))

инвертируем бинарную маску (255 - B) и ищем связные области функцией connectedComponentsWithStats

n = 10 — всего 10 меток с учётом фона. Метка 0 — нулевые пиксели после инверсии (сами шестерёнки), метка 1 — внешняя область; метки 2–9 — 8 отверстий.

In [ ]:
n, labels, stats, centroids = cv2.connectedComponentsWithStats(255 - B)
print(n)
print(stats)

Теперь ищем сами шестеренки

n = 3: метка 0 — фон, метки 1 и 2 — две шестерёнки. Их ограничивающие прямоугольники — примерно 311 × 306 и 312 × 309 px.

In [ ]:
n, labels, stats, centroids = cv2.connectedComponentsWithStats(B)
print(n)
print(stats)

создаём кольцо make_ring(101, 2) и выполняем эрозию B1 = erode(B, ring)

8 связных областей — по одной возле каждого отверстия. На рисунке B1 белые области показаны поверх затемнённой маски B // 4.

In [ ]:
ring = make_ring(101, 2)

plt.figure(figsize=(3, 3))
plt.imshow(ring, cmap="gray")
plt.show()

B1 = cv2.erode(B, ring)
n, _ = cv2.connectedComponents(B1)
# Найдено 8 связных областей, соответствующих восьми отверстиям шестерёнок.
# Вычитаем единицу, потому что connectedComponents учитывает фон как отдельную метку.
print(n - 1)

show(cv2.add(B // 4, B1), "B1") # // 4 делает шестеренки серыми

расширяем B1 диском 91 px и объединяем B2 с исходной маской через bitwise_or, получая B3

In [ ]:
B2 = cv2.dilate(B1, disk(91))
B3 = cv2.bitwise_or(B, B2)

n, _ = cv2.connectedComponents(255 - B3)
print(n)

show(B2, "B2")
show(B3, "B3")

Подбор диаметра диска
проверяем эрозию B3 дисками диаметром
251–289 px. Метод max() показывает, остались ли белые
пиксели после эрозии.
при 285 px белые пиксели ещё остаются,
при 289 px исчезают. Выбираем 261 px для устойчивого
выделения центральных областей.

In [ ]:
for d in [251, 261, 271, 281, 285, 289]:
    print(d, cv2.erode(B3, disk(d)).max())

применяем эрозию B3 диском диаметром
261 px. После эрозии остаются две небольшие белые
области около центров шестерёнок. Нам нужны эти области чтобы построить измерительные кольца в области зубьев
2 связные компоненты — по одной
на каждую шестерёнку.

In [ ]:
B4 = cv2.erode(B3, disk(261))
n, _ = cv2.connectedComponents(B4)
print(n - 1)

show(cv2.add(B3 // 4, B4), "B4")

строим измерительные кольца, проходящие
через зубья шестерёнок. Далее по пересечениям
колец с зубьями будем искать повреждения.

In [ ]:
B5 = cv2.dilate(B4, disk(269))
B6 = cv2.dilate(B5, disk(7))
B7 = cv2.subtract(B6, B5)

show(B5, "B5")
show(B7, "B7")
show(cv2.add(B // 4, B7), "B7")

оставляем только те пиксели измерительных колец, которые совпадают с белыми пикселями исходных шестерёнок. получаем короткие белые участки в местах, где кольцо пересекает имеющиеся зубья. B8 показывает, в каких местах измерительного кольца присутствует металл зубьев.

важно! n = 124 это число отдельных фрагментов пересечения, не обязательно точное количество зубьев: из-за толщины кольца и дискретных пикселей один зуб может дать несколько фрагментов.

In [ ]:
B8 = cv2.bitwise_and(B, B7)
n, _ = cv2.connectedComponents(B8)
print(n - 1)

show(B8, "B8")

для диаметров d = 9, 11, ..., 29 расширяем фрагменты B8 вычитаем их из B7 и считаем оставшиеся компоненты D.
при d = 9 остаётся 81 разрыв, при 11 — 13; при 13–23 — 2 разрыва, а при 25–29 — 0.

In [ ]:
rows = []
for d in range(9, 31, 2):
    B9 = cv2.dilate(B8, disk(d))
    D = cv2.subtract(B7, B9)
    n, _ = cv2.connectedComponents(D)
    rows.append({"tip_spacing": d, "defects": n - 1})

pd.DataFrame(rows).set_index("tip_spacing")

перебираем диаметр расширения центральных областей s = 259, 261, ..., 275 px. Каждый раз заново строим B5–B8, расширяем срезы зубьев диском 17 px и считаем оставшиеся дефекты.

при 259–261 обнаруживается 0 разрывов, при 263–265 — 1, при 267–273 — 2, а при 275 — 4.

In [ ]:
rows = []
for s in range(259, 277, 2):
    B5 = cv2.dilate(B4, disk(s))
    B6 = cv2.dilate(B5, disk(7))
    B7 = cv2.subtract(B6, B5)
    B8 = cv2.bitwise_and(B, B7)
    B9 = cv2.dilate(B8, disk(17))
    D = cv2.subtract(B7, B9)
    n, _ = cv2.connectedComponents(D)
    rows.append({"spacer": s, "defects": n - 1})

pd.DataFrame(rows).set_index("spacer")

заново строим B5–B8 с диаметрами 269 и 7 px, расширяем пересечения зубьев диском 17 px (B9) и вычисляем D = B7 - B9.
269 задаёт положение кольца, 7 — его небольшую толщину, 17 — допустимую величину обычных промежутков между срезами зубьев.

In [ ]:
B5 = cv2.dilate(B4, disk(269))
B6 = cv2.dilate(B5, disk(7))
B7 = cv2.subtract(B6, B5)
B8 = cv2.bitwise_and(B, B7)
B9 = cv2.dilate(B8, disk(17))
D = cv2.subtract(B7, B9)

n, _ = cv2.connectedComponents(D)
print(n - 1)

show(B9, "B9")
show(D, "B7 - B9")

расширяем небольшие дефектные области D диском 21 px (D2) и объединяем их с B9 через bitwise_or, получая RESULT.

In [ ]:
D2 = cv2.dilate(D, disk(21))
RESULT = cv2.bitwise_or(D2, B9)

show(RESULT, "RESULT")
show(cv2.add(B9 // 2, D2), "RESULT")

считаем связные области в D2, красим измерительные кольца на исходном RGB-изображении в зелёный цвет, дефекты — в красный, а вокруг каждого дефекта рисуем прямоугольную рамку.

2 дефекта. Центры их ограничивающих прямоугольников примерно (664, 242) и (183, 369)

In [ ]:
count, masks = cv2.connectedComponents(D2)
print(count - 1)

vis = img.copy()
vis[B7 > 0] = (0, 200, 0)
vis[D2 > 0] = (255, 0, 0)

for mask_label in range(1, count):
    target_mask = masks == mask_label
    x1, y1, w, h = cv2.boundingRect(target_mask.astype(np.uint8) * 255)
    x2 = x1 + w
    y2 = y1 + h
    print(x1 + w // 2, y1 + h // 2)
    vis = cv2.rectangle(vis, (x1 - 5, y1 - 5), (x2 + 5, y2 + 5), (255, 0, 0), 2, cv2.LINE_AA)

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(vis)
plt.show()

In [ ]:
vis2 = np.dstack([B // 4] * 3)
vis2[B7 > 0] = (0, 150, 0)
vis2[B8 > 0] = (255, 255, 255)
vis2[D > 0] = (255, 0, 0)

for mask_label in range(1, count):
    ys, xs = np.where(masks == mask_label)
    x, y = int(xs.mean()), int(ys.mean())

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(B[y-30:y+30, x-30:x+30], cmap="gray")
    axes[1].imshow(vis2[y-30:y+30, x-30:x+30])
    plt.show()